# 001. Air you can see: the OpenAQ semantic model, star schema first

Code from post 001 of the 365 days of Microsoft Fabric series, in the order the post runs it. The post explains each cell. The post publishes on t-sql.dk on Monday 9 November 2026.

*Status: runnable design. The archive and pandas steps below have been run against the public OpenAQ files. The Fabric steps (Delta tables, semantic model and the three checks) have not been run in my workspace yet, and their output goes into this post before it is published.*

In [ ]:
%pip install semantic-link-labs

In [ ]:
# Parameters
VAULT_URL = "https://<your-vault-name>.vault.azure.net/"
LOCATION_IDS = [2993, 10516]  # Berlin Neukölln, Kraków ul. Bulwarowa
YEAR, MONTH = 2026, 9
LAKEHOUSE = "lh_openaq"  # attached as the default lakehouse
MODEL_NAME = "OpenAQ air quality"

In [ ]:
import time
import requests

API = "https://api.openaq.org/v3"
session = requests.Session()
session.headers["X-API-Key"] = notebookutils.credentials.getSecret(VAULT_URL, "openaq-api-key")

def openaq_get(path):
    response = session.get(f"{API}{path}", timeout=30)
    if response.status_code == 429:
        time.sleep(int(response.headers.get("x-ratelimit-reset", "60")) + 1)
        response = session.get(f"{API}{path}", timeout=30)
    response.raise_for_status()
    time.sleep(1)
    return response.json()["results"]

locations = [openaq_get(f"/locations/{location_id}")[0] for location_id in LOCATION_IDS]

In [ ]:
import io
import xml.etree.ElementTree as ET
import pandas as pd

ARCHIVE = "https://openaq-data-archive.s3.amazonaws.com"
S3_NS = {"s3": "http://s3.amazonaws.com/doc/2006-03-01/"}

def list_archive_keys(location_id, year, month):
    prefix = f"records/csv.gz/locationid={location_id}/year={year}/month={month:02d}/"
    response = requests.get(ARCHIVE, params={"list-type": "2", "prefix": prefix}, timeout=30)
    response.raise_for_status()
    root = ET.fromstring(response.content)
    return [node.text for node in root.findall("s3:Contents/s3:Key", S3_NS)]

frames = []
for location_id in LOCATION_IDS:
    for key in list_archive_keys(location_id, YEAR, MONTH):
        response = requests.get(f"{ARCHIVE}/{key}", timeout=30)
        response.raise_for_status()
        frame = pd.read_csv(io.BytesIO(response.content), compression="gzip", dtype=str)
        frame["SourceFile"] = key
        frames.append(frame)

raw = pd.concat(frames, ignore_index=True)
print(f"{raw['SourceFile'].nunique()} files, {len(raw)} rows")

In [ ]:
dim_location = pd.DataFrame([{
    "LocationId": loc["id"],
    "LocationName": loc["name"],
    "Locality": loc["locality"],
    "CountryCode": loc["country"]["code"],
    "Timezone": loc["timezone"],
    "Latitude": loc["coordinates"]["latitude"],
    "Longitude": loc["coordinates"]["longitude"],
    "ProviderName": loc["provider"]["name"],
    "OwnerName": loc["owner"]["name"],
    "IsMonitor": loc["isMonitor"],
} for loc in locations])

dim_parameter = pd.DataFrame([{
    "ParameterId": sensor["parameter"]["id"],
    "ParameterName": sensor["parameter"]["name"],
    "DisplayName": sensor["parameter"]["displayName"],
    "Unit": sensor["parameter"]["units"],
} for loc in locations for sensor in loc["sensors"]]).drop_duplicates()
assert dim_parameter["ParameterId"].is_unique, "One parameter ID with two descriptions"

sensor_parameter = pd.DataFrame([
    {"SensorId": sensor["id"], "ParameterId": sensor["parameter"]["id"]}
    for loc in locations for sensor in loc["sensors"]
])

ref_guideline = pd.DataFrame([
    {"ParameterName": "pm25", "AveragingPeriod": "24-hour", "GuidelineValue": 15.0, "Unit": "µg/m³",
     "Edition": "WHO 2021", "SourceUrl": "https://www.who.int/publications/i/item/9789240034228"},
    {"ParameterName": "pm25", "AveragingPeriod": "Annual", "GuidelineValue": 5.0, "Unit": "µg/m³",
     "Edition": "WHO 2021", "SourceUrl": "https://www.who.int/publications/i/item/9789240034228"},
])

In [ ]:
local_text = raw["datetime"]
measurement_utc = pd.to_datetime(local_text, utc=True)
# The archive stamps the end of the hour, so the 00:00 reading closes the previous day.
local_hour_start = pd.to_datetime(local_text.str[:19]) - pd.Timedelta(hours=1)

fact = pd.DataFrame({
    "LocationId": raw["location_id"].astype("int64"),
    "SensorId": raw["sensors_id"].astype("int64"),
    "DateKey": local_hour_start.dt.strftime("%Y%m%d").astype("int32"),
    "MeasurementUtc": measurement_utc.dt.tz_localize(None),
    "MeasurementLocal": local_text,
    "Value": raw["value"].astype("float64"),
    "SourceUnit": raw["units"],
    "SourceFile": raw["SourceFile"],
}).merge(sensor_parameter, on="SensorId", how="left")

unmapped = fact[fact["ParameterId"].isna()]
units = fact.merge(dim_parameter[["ParameterId", "Unit"]], on="ParameterId", how="left")
unit_mismatch = units[units["SourceUnit"] != units["Unit"]]
duplicates = int(fact.duplicated(subset=["SensorId", "MeasurementUtc"], keep=False).sum())
print(f"Unmapped sensors: {len(unmapped)}, unit mismatches: {len(unit_mismatch)}, "
      f"duplicate sensor hours: {duplicates}")
assert unmapped.empty and unit_mismatch.empty, "Look at these rows before loading anything"
fact["ParameterId"] = fact["ParameterId"].astype("int64")

In [ ]:
first = pd.to_datetime(str(fact["DateKey"].min()), format="%Y%m%d")
last = pd.to_datetime(str(fact["DateKey"].max()), format="%Y%m%d")
dates = pd.date_range(first.replace(day=1), last + pd.offsets.MonthEnd(0), freq="D")
dim_date = pd.DataFrame({
    "DateKey": dates.strftime("%Y%m%d").astype("int32"),
    "Date": dates.date,
    "Year": dates.year,
    "MonthNumber": dates.month,
    "MonthName": dates.strftime("%B"),
    "YearMonth": dates.strftime("%Y-%m"),
    "DayOfMonth": dates.day,
})

In [ ]:
spark.conf.set("spark.sql.session.timeZone", "UTC")

SCHEMAS = {
    "dim_location": "LocationId bigint, LocationName string, Locality string, CountryCode string, "
                    "Timezone string, Latitude double, Longitude double, ProviderName string, "
                    "OwnerName string, IsMonitor boolean",
    "dim_parameter": "ParameterId bigint, ParameterName string, DisplayName string, Unit string",
    "dim_date": "DateKey int, Date date, Year int, MonthNumber int, MonthName string, "
                "YearMonth string, DayOfMonth int",
    "ref_guideline": "ParameterName string, AveragingPeriod string, GuidelineValue double, "
                     "Unit string, Edition string, SourceUrl string",
    "fact_measurement": "LocationId bigint, ParameterId bigint, DateKey int, SensorId bigint, "
                        "MeasurementUtc timestamp, MeasurementLocal string, Value double, "
                        "SourceUnit string, SourceFile string",
}
FACT_COLUMNS = ["LocationId", "ParameterId", "DateKey", "SensorId", "MeasurementUtc",
                "MeasurementLocal", "Value", "SourceUnit", "SourceFile"]
tables = {
    "dim_location": dim_location,
    "dim_parameter": dim_parameter,
    "dim_date": dim_date,
    "ref_guideline": ref_guideline,
    "fact_measurement": fact[FACT_COLUMNS],
}
for name, frame in tables.items():
    (spark.createDataFrame(frame, SCHEMAS[name]).write.format("delta").mode("overwrite")
        .option("overwriteSchema", "true").saveAsTable(name))

In [ ]:
from sempy_labs import refresh_semantic_model
from sempy_labs.directlake import generate_direct_lake_semantic_model
from sempy_labs.tom import connect_semantic_model

generate_direct_lake_semantic_model(
    dataset=MODEL_NAME, lakehouse_tables=list(tables), lakehouse=LAKEHOUSE,
    overwrite=True, refresh=False,
)

with connect_semantic_model(dataset=MODEL_NAME, readonly=False) as tom:
    for dimension, key in [("dim_location", "LocationId"), ("dim_parameter", "ParameterId"),
                           ("dim_date", "DateKey")]:
        tom.add_relationship(
            from_table="fact_measurement", from_column=key, to_table=dimension, to_column=key,
            from_cardinality="Many", to_cardinality="One", cross_filtering_behavior="OneDirection",
        )
        tom.model.Tables["fact_measurement"].Columns[key].IsHidden = True
    tom.mark_as_date_table(table_name="dim_date", column_name="Date")
    month_name = tom.model.Tables["dim_date"].Columns["MonthName"]
    month_name.SortByColumn = tom.model.Tables["dim_date"].Columns["MonthNumber"]

refresh_semantic_model(dataset=MODEL_NAME)

In [ ]:
for loc in locations:
    api_sensors = {sensor["id"] for sensor in loc["sensors"]}
    archive_sensors = set(fact.loc[fact["LocationId"] == loc["id"], "SensorId"])
    print(loc["name"],
          "| archive only:", sorted(archive_sensors - api_sensors),
          "| API only:", sorted(api_sensors - archive_sensors))

In [ ]:
row = spark.sql("""
    SELECT f.* FROM fact_measurement f
    JOIN dim_parameter p ON p.ParameterId = f.ParameterId
    WHERE p.ParameterName = 'pm25'
    ORDER BY f.MeasurementUtc LIMIT 1
""").first()
source = pd.read_csv(f"{ARCHIVE}/{row.SourceFile}", compression="gzip", dtype=str)
print(row)
print(source[(source["sensors_id"] == str(row.SensorId))
             & (source["datetime"] == row.MeasurementLocal)])

In [ ]:
import sempy.fabric as fabric

lakehouse_rows = spark.table("fact_measurement").count()
result = fabric.evaluate_dax(MODEL_NAME, f"""
EVALUATE
VAR TotalRows = COUNTROWS ( fact_measurement )
VAR SumOverLocations =
    SUMX ( VALUES ( dim_location[LocationId] ), CALCULATE ( COUNTROWS ( fact_measurement ) ) )
RETURN
    ROW ( "Total", TotalRows, "Sum over locations", SumOverLocations, "Lakehouse", {lakehouse_rows} )
""")
display(result)